# # PET REMOTE SYSTEM_(3) AUTOML2 + 누수 제거

In [ ]:
#!/usr/bin/env python3
"""
반려동물 건강 상태 이진분류 AutoML 스크립트 (데이터 누수 방지)
사용법: python pet_automl_realistic.py [csv_파일_경로]
예시: python pet_automl_realistic.py fixed_pet_data_1050.csv
"""

import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                            accuracy_score, precision_score, recall_score, f1_score, make_scorer)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

class PetHealthRealisticAutoML:
    def __init__(self, csv_path):
        self.csv_path = "fixed_pet_data_1050.csv"
        self.df = None
        self.best_model = None
        self.results = {}
        
    def load_and_prepare_data(self):
        """데이터 로드 및 이진분류 준비"""
        print("📊 ===== 이진분류 데이터 준비 =====")
        
        try:
            self.df = pd.read_csv(self.csv_path)
            print(f"✅ 데이터 로드: {self.df.shape}")
        except FileNotFoundError:
            print(f"❌ 파일을 찾을 수 없습니다: {self.csv_path}")
            sys.exit(1)
        except Exception as e:
            print(f"❌ 데이터 로드 실패: {e}")
            sys.exit(1)
        
        # severity 컬럼 확인
        if 'severity' not in self.df.columns:
            print(f"❌ 'severity' 컬럼이 없습니다. 컬럼 목록: {list(self.df.columns)}")
            sys.exit(1)
        
        # 이진분류 타겟 생성
        self.df['binary_target'] = (self.df['severity'] > 0).astype(int)
        
        # 클래스 분포 확인
        binary_dist = self.df['binary_target'].value_counts()
        total = len(self.df)
        
        print(f"🎯 이진분류 타겟 분포:")
        print(f"   정상 (0): {binary_dist[0]:3d}개 ({binary_dist[0]/total*100:5.1f}%)")
        print(f"   질병 (1): {binary_dist[1]:3d}개 ({binary_dist[1]/total*100:5.1f}%)")
        
        # 불균형 비율
        imbalance_ratio = binary_dist[0] / binary_dist[1]
        print(f"📈 불균형 비율: {imbalance_ratio:.2f}:1")
        
        if imbalance_ratio < 1.5:
            print(f"   ✅ 매우 균형적! 특별한 처리 불필요")
            self.imbalance_strategy = 'none'
        elif imbalance_ratio < 3.0:
            print(f"   ⚠️ 경미한 불균형 - 가중치 조정 적용")
            self.imbalance_strategy = 'weight'
        else:
            print(f"   ❌ 심각한 불균형 - 가중치 조정 적용")
            self.imbalance_strategy = 'weight'
        
        return self.df
    
    def feature_selection_and_cleaning(self):
        """특성 선택 및 정리 (데이터 누수 방지)"""
        print("\n🔧 ===== 특성 선택 및 정리 (데이터 누수 방지) =====")
        
        # 제외할 컬럼들
        exclude_cols = ['pet_id', 'severity', 'binary_target']
        
        # 🚨 데이터 누수 의심 특성들 강제 제거
        suspected_leakage_cols = [
            'medical_value_0', 'medical_value_1',  # 의료 결과값 (타겟과 거의 동일)
            'medical_foot_position_0', 'medical_foot_position_1'  # 진단 관련 정보
        ]
        
        # 실제 존재하는 누수 특성만 제외 목록에 추가
        existing_leakage_cols = [col for col in suspected_leakage_cols if col in self.df.columns]
        if existing_leakage_cols:
            exclude_cols.extend(existing_leakage_cols)
            print(f"🚨 데이터 누수 의심 특성 제거: {len(existing_leakage_cols)}개")
            for col in existing_leakage_cols:
                # 상관관계 확인해서 출력
                if col in self.df.columns:
                    corr_val = self.df[col].corr(self.df['binary_target'])
                    print(f"   - {col}: 상관관계 {corr_val:.3f} (너무 높음!)")
            print(f"   → 이런 특성들은 실제 예측 시 사용할 수 없는 정보입니다")
        
        feature_cols = [col for col in self.df.columns if col not in exclude_cols]
        print(f"📊 전체 특성 수 (누수 제거 후): {len(feature_cols)}")
        
        # 높은 결측값 특성 제거 (80% 이상)
        missing_pct = self.df[feature_cols].isnull().sum() / len(self.df)
        high_missing_cols = missing_pct[missing_pct > 0.8].index.tolist()
        
        if high_missing_cols:
            print(f"❌ 80% 이상 결측 특성 제거: {len(high_missing_cols)}개")
            for col in high_missing_cols[:5]:
                print(f"   - {col}: {missing_pct[col]*100:.1f}% 결측")
            feature_cols = [col for col in feature_cols if col not in high_missing_cols]
        
        # 상관관계 기반 특성 순위 (현실적 범위로 필터링)
        numeric_cols = [col for col in feature_cols if self.df[col].dtype in ['int64', 'float64']]
        
        if numeric_cols:
            correlations = self.df[numeric_cols].corrwith(self.df['binary_target']).abs().sort_values(ascending=False)
            
            print(f"\n📊 이진분류 타겟과 상관관계 TOP 15 (현실적 범위):")
            for i, (feature, corr) in enumerate(correlations.head(15).items()):
                bar = "█" * int(corr * 20)
                # 데이터 누수 의심 표시
                warning = " ⚠️" if corr > 0.7 else ""
                print(f"   {i+1:2d}. {feature[:30]:30s}: {corr:.3f} {bar}{warning}")
            
            # 🎯 현실적인 상관관계 범위로 특성 선택
            # 너무 높은 상관관계(0.7+) = 데이터 누수 의심
            # 너무 낮은 상관관계(0.02-) = 예측력 부족
            realistic_features = correlations[
                (correlations >= 0.02) & (correlations < 0.7)
            ].index.tolist()
            
            # 너무 적으면 상위 25개 선택 (0.7 미만)
            if len(realistic_features) < 10:
                safe_features = correlations[correlations < 0.7]
                realistic_features = safe_features.head(25).index.tolist()
                print(f"   ⚠️ 현실적 범위 특성이 적어서 상위 25개 선택")
            elif len(realistic_features) > 40:
                realistic_features = realistic_features[:40]
                print(f"   ✂️ 너무 많아서 상위 40개로 제한")
            
            print(f"✅ 선택된 현실적 특성: {len(realistic_features)}개")
            print(f"   → 상관관계 범위: 0.02 ≤ r < 0.70 (데이터 누수 방지)")
            
            # 범주형 특성 추가
            categorical_cols = [col for col in feature_cols if self.df[col].dtype == 'object']
            final_features = realistic_features + categorical_cols
            
        else:
            final_features = feature_cols
        
        print(f"🎯 최종 특성 수: {len(final_features)}")
        print(f"🏥 의료적 현실성: 예상 성능 80-95% (100% 방지)")
        
        # 특성 데이터 준비
        X = self.df[final_features].copy()
        y = self.df['binary_target'].copy()
        
        # 범주형 변수 인코딩
        categorical_cols = [col for col in final_features if self.df[col].dtype == 'object']
        if categorical_cols:
            for col in categorical_cols:
                if col in X.columns:
                    le = LabelEncoder()
                    X[col] = le.fit_transform(X[col].astype(str))
                    print(f"   ✅ {col} 라벨 인코딩 완료")
        
        # 결측값 처리
        numeric_cols_final = X.select_dtypes(include=[np.number]).columns
        X[numeric_cols_final] = X[numeric_cols_final].fillna(X[numeric_cols_final].median())
        
        print(f"✅ 데이터 준비 완료: X{X.shape}, y{y.shape}")
        
        return X, y, final_features
    
    def run_automl_comparison(self, X, y):
        """다양한 ML 모델 자동 비교"""
        print("\n🤖 ===== AutoML 모델 비교 시작 =====")
        
        # 데이터 분할
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        print(f"📊 데이터 분할:")
        print(f"   훈련: {X_train.shape}, 테스트: {X_test.shape}")
        print(f"   훈련 클래스 분포: {pd.Series(y_train).value_counts().to_dict()}")
        
        # 클래스 가중치 설정
        class_weight = 'balanced' if self.imbalance_strategy == 'weight' else None
        
        # 기본 모델들
        models = {
            'Random Forest': RandomForestClassifier(
                n_estimators=100, random_state=42, 
                class_weight=class_weight, n_jobs=-1
            ),
            'Gradient Boosting': GradientBoostingClassifier(
                n_estimators=100, random_state=42
            ),
            'Logistic Regression': LogisticRegression(
                random_state=42, class_weight=class_weight, max_iter=1000
            ),
            'SVM': SVC(
                random_state=42, class_weight=class_weight, probability=True
            ),
            'Naive Bayes': GaussianNB(),
            'K-Neighbors': KNeighborsClassifier(n_neighbors=5)
        }
        
        # XGBoost 추가 (설치되어 있다면)
        try:
            import xgboost as xgb
            if self.imbalance_strategy == 'weight':
                scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
            else:
                scale_pos_weight = 1
                
            models['XGBoost'] = xgb.XGBClassifier(
                n_estimators=100, random_state=42,
                scale_pos_weight=scale_pos_weight, eval_metric='logloss'
            )
            print("✅ XGBoost 추가됨")
        except ImportError:
            print("⚠️ XGBoost 미설치 - pip install xgboost로 설치 권장")
        
        # LightGBM 추가 (설치되어 있다면)
        try:
            import lightgbm as lgb
            models['LightGBM'] = lgb.LGBMClassifier(
                n_estimators=100, random_state=42,
                class_weight=class_weight, verbosity=-1
            )
            print("✅ LightGBM 추가됨")
        except ImportError:
            print("⚠️ LightGBM 미설치 - pip install lightgbm으로 설치 권장")
        
        # 모델 성능 비교
        results = {}
        
        print(f"\n🔄 모델 훈련 및 평가 중...")
        
        # 의료 AI 중요 지표들
        scoring_metrics = {
            'accuracy': 'accuracy',
            'precision': make_scorer(precision_score, pos_label=1),
            'recall': make_scorer(recall_score, pos_label=1),  # 질병 놓치지 않기
            'f1': make_scorer(f1_score, pos_label=1),
            'roc_auc': 'roc_auc'
        }
        
        for model_name, model in models.items():
            print(f"   🔄 {model_name} 평가 중...")
            
            model_results = {}
            
            # 교차검증으로 성능 평가
            for metric_name, scorer in scoring_metrics.items():
                try:
                    scores = cross_val_score(model, X_train, y_train, cv=5, scoring=scorer, n_jobs=-1)
                    model_results[metric_name] = {
                        'mean': scores.mean(),
                        'std': scores.std()
                    }
                except Exception as e:
                    model_results[metric_name] = {'mean': 0, 'std': 0}
            
            # 실제 훈련 및 테스트 성능
            try:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
                
                # 테스트 성능
                model_results['test_accuracy'] = accuracy_score(y_test, y_pred)
                model_results['test_precision'] = precision_score(y_test, y_pred)
                model_results['test_recall'] = recall_score(y_test, y_pred)
                model_results['test_f1'] = f1_score(y_test, y_pred)
                
                if y_pred_proba is not None:
                    model_results['test_roc_auc'] = roc_auc_score(y_test, y_pred_proba)
                
                # 특성 중요도 (가능한 경우)
                if hasattr(model, 'feature_importances_'):
                    feature_importance = dict(zip(X.columns, model.feature_importances_))
                    model_results['feature_importance'] = feature_importance
                
                results[model_name] = model_results
                
            except Exception as e:
                print(f"     ❌ {model_name} 훈련 실패: {e}")
        
        print(f"✅ {len(results)}개 모델 평가 완료")
        
        return results, X_train, X_test, y_train, y_test
    
    def analyze_results_and_select_best(self, results):
        """결과 분석 및 최적 모델 선택"""
        print(f"\n📊 ===== 모델 성능 비교 결과 (현실적 범위) =====")
        
        # 결과 정리
        comparison_df = []
        
        for model_name, model_results in results.items():
            row = {'Model': model_name}
            
            # 교차검증 점수들
            for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
                if metric in model_results:
                    row[f'CV_{metric}'] = f"{model_results[metric]['mean']:.3f} ±{model_results[metric]['std']:.3f}"
                    row[f'{metric}_mean'] = model_results[metric]['mean']
            
            # 테스트 점수들
            for metric in ['test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
                if metric in model_results:
                    row[metric] = model_results[metric]
            
            comparison_df.append(row)
        
        comparison_df = pd.DataFrame(comparison_df)
        
        # 성능 순위표 출력
        print(f"🏆 모델 성능 순위 (테스트 세트 기준):")
        print(f"{'Model':<15} {'Accuracy':<8} {'Precision':<9} {'Recall':<8} {'F1':<8} {'ROC-AUC':<8}")
        print(f"-" * 70)
        
        for _, row in comparison_df.iterrows():
            model_name = row['Model'][:14]
            acc = row.get('test_accuracy', 0)
            prec = row.get('test_precision', 0)
            rec = row.get('test_recall', 0)
            f1 = row.get('test_f1', 0)
            auc = row.get('test_roc_auc', 0)
            
            print(f"{model_name:<15} {acc:<8.3f} {prec:<9.3f} {rec:<8.3f} {f1:<8.3f} {auc:<8.3f}")
        
        # 의료 AI 관점에서 최적 모델 선택
        print(f"\n🏥 ===== 의료 AI 관점 모델 선택 =====")
        
        # 가중 점수 계산 (의료에서 중요한 지표들)
        weights = {
            'test_recall': 0.35,    # 질병 놓치지 않기 (가장 중요)
            'test_precision': 0.25, # 오진 최소화
            'test_f1': 0.25,        # 균형
            'test_roc_auc': 0.15    # 전체 성능
        }
        
        comparison_df['medical_score'] = 0
        for metric, weight in weights.items():
            if metric in comparison_df.columns:
                comparison_df['medical_score'] += comparison_df[metric] * weight
        
        # 점수별 정렬
        comparison_df = comparison_df.sort_values('medical_score', ascending=False)
        
        print(f"📊 의료 AI 특화 점수 순위:")
        for i, (_, row) in enumerate(comparison_df.head(5).iterrows()):
            medal = ["🥇", "🥈", "🥉", "4️⃣", "5️⃣"][i]
            print(f"   {medal} {row['Model']}: {row['medical_score']:.3f}점")
        
        # 최고 모델 선택
        best_model_name = comparison_df.iloc[0]['Model']
        best_model_results = results[best_model_name]
        
        print(f"\n🎉 ===== 선택된 최고 모델: {best_model_name} =====")
        print(f"📊 상세 성능:")
        print(f"   정확도 (Accuracy): {best_model_results.get('test_accuracy', 0):.3f}")
        print(f"   정밀도 (Precision): {best_model_results.get('test_precision', 0):.3f}")
        print(f"   재현율 (Recall): {best_model_results.get('test_recall', 0):.3f}")
        print(f"   F1 점수: {best_model_results.get('test_f1', 0):.3f}")
        if 'test_roc_auc' in best_model_results:
            print(f"   ROC-AUC: {best_model_results['test_roc_auc']:.3f}")
        
        # 현실성 검증
        acc_score = best_model_results.get('test_accuracy', 0)
        if acc_score >= 0.99:
            print(f"\n⚠️ 경고: 정확도 {acc_score:.1%}는 의료 데이터치고 너무 높습니다!")
            print(f"   → 숨겨진 데이터 누수가 있을 수 있습니다")
        elif 0.80 <= acc_score <= 0.95:
            print(f"\n✅ 현실적 성능: {acc_score:.1%} (의료 AI 적절 범위)")
        
        # 특성 중요도 (있는 경우)
        if 'feature_importance' in best_model_results:
            print(f"\n📊 주요 특성 중요도 TOP 15:")
            importance_sorted = sorted(
                best_model_results['feature_importance'].items(), 
                key=lambda x: x[1], reverse=True
            )
            
            for i, (feature, importance) in enumerate(importance_sorted[:15]):
                bar = "█" * int(importance * 50)
                # 포즈/센서 특성 구분
                if any(sensor in feature for sensor in ['P0_', 'P1_', 'P2_', 'P3_', 'P4_', 'P5_', 'P6_']):
                    feature_type = "🦴"
                elif 'sensor' in feature:
                    feature_type = "📱"
                elif feature in ['dog_type', 'age_numeric', 'size_numeric']:
                    feature_type = "🐕"
                else:
                    feature_type = "⚙️"
                
                print(f"   {i+1:2d}. {feature_type} {feature[:23]:23s}: {importance:.3f} {bar}")
        
        return best_model_name, best_model_results, comparison_df
    
    def run_complete_automl(self):
        """전체 AutoML 파이프라인 실행 (데이터 누수 방지)"""
        print("🚀 ===== 반려동물 질병 이진분류 AutoML 시작 (현실적 버전) =====")
        print(f"📁 데이터 파일: {self.csv_path}")
        
        # 1. 데이터 준비
        self.load_and_prepare_data()
        
        # 2. 특성 선택 (데이터 누수 방지)
        X, y, features = self.feature_selection_and_cleaning()
        
        # 3. 모델 비교
        results, X_train, X_test, y_train, y_test = self.run_automl_comparison(X, y)
        
        # 4. 최적 모델 선택
        best_model_name, best_results, comparison_df = self.analyze_results_and_select_best(results)
        
        # 5. 결과 저장
        output_file = f'automl_realistic_results_{self.csv_path.split(".")[0]}.csv'
        comparison_df.to_csv(output_file, index=False)
        print(f"\n💾 결과 저장: {output_file}")
        
        # 6. 최종 권장사항 (현실적 기준)
        print(f"\n💡 ===== 최종 권장사항 (의료 AI 현실 기준) =====")
        
        recall_score = best_results.get('test_recall', 0)
        precision_score = best_results.get('test_precision', 0)
        accuracy_score = best_results.get('test_accuracy', 0)
        
        if accuracy_score >= 0.98:
            print(f"🚨 비현실적 성능! 추가 검토 필요")
            print(f"   → {accuracy_score:.1%} 정확도는 의료 데이터치고 의심스럽습니다")
            print(f"   → 숨겨진 데이터 누수나 과적합 가능성")
            
        elif recall_score >= 0.85 and precision_score >= 0.75 and 0.80 <= accuracy_score <= 0.95:
            print(f"🎉 우수한 현실적 성능! 실용화 검토 가능")
            print(f"   - 질병 탐지율: {recall_score:.1%} (85% 이상 권장)")
            print(f"   - 정밀도: {precision_score:.1%} (75% 이상 권장)")
            print(f"   - 전체 정확도: {accuracy_score:.1%} (의료 AI 적절 범위)")
            
        elif recall_score >= 0.75 and precision_score >= 0.70:
            print(f"👍 양호한 성능! 개선 후 실용화 검토")
            print(f"   - 추가 특성 엔지니어링 고려")
            print(f"   - 하이퍼파라미터 튜닝 권장")
            print(f"   - 더 많은 데이터 수집 고려")
            
        else:
            print(f"⚠️ 성능 개선 필요")
            print(f"   - 현재 성능: 정확도 {accuracy_score:.1%}, 재현율 {recall_score:.1%}")
            print(f"   - 더 많은 데이터 수집")
            print(f"   - 특성 엔지니어링 재검토")
            print(f"   - 다른 알고리즘 시도")
        
        print(f"\n🎯 포즈/센서 데이터만으로 달성한 순수 AI 성능입니다!")
        
        return {
            'best_model': best_model_name,
            'best_results': best_results,
            'all_results': results,
            'comparison_df': comparison_df,
            'data': (X_train, X_test, y_train, y_test)
        }

def main():
    """메인 실행 함수"""
    print("🤖 반려동물 건강 상태 AutoML v2.0 (현실적 버전)")
    print("=" * 55)
    
    # 명령행 인수 처리
    if len(sys.argv) > 1:
        csv_path = sys.argv[1]
    else:
        # 기본 파일명들 시도
        default_files = ['fixed_pet_data_1050.csv', 'pet_data.csv', 'data.csv']
        csv_path = None
        
        for file in default_files:
            try:
                pd.read_csv(file, nrows=1)  # 파일 존재 확인
                csv_path = file
                print(f"📁 자동 발견된 파일: {file}")
                break
            except:
                continue
        
        if csv_path is None:
            print("❌ CSV 파일을 찾을 수 없습니다.")
            print("사용법: python pet_automl_realistic.py [csv_파일_경로]")
            print("예시: python pet_automl_realistic.py fixed_pet_data_1050.csv")
            sys.exit(1)
    
    # AutoML 실행
    try:
        automl = PetHealthRealisticAutoML(csv_path)
        results = automl.run_complete_automl()
        
        print(f"\n🎯 ===== 최종 결과 요약 =====")
        print(f"🏆 최고 모델: {results['best_model']}")
        print(f"📊 현실적 성능:")
        best = results['best_results']
        print(f"   정확도: {best.get('test_accuracy', 0):.1%}")
        print(f"   질병탐지율: {best.get('test_recall', 0):.1%}")
        print(f"   정밀도: {best.get('test_precision', 0):.1%}")
        print(f"   F1 점수: {best.get('test_f1', 0):.1%}")
        
        print(f"\n✅ 현실적 AutoML 완료! 포즈/센서 데이터만의 순수 AI 성능입니다.")
        
    except Exception as e:
        print(f"❌ AutoML 실행 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

if __name__ == "__main__":
    main()

🤖 반려동물 건강 상태 AutoML v2.0 (현실적 버전)
🚀 ===== 반려동물 질병 이진분류 AutoML 시작 (현실적 버전) =====
📁 데이터 파일: fixed_pet_data_1050.csv
📊 ===== 이진분류 데이터 준비 =====
✅ 데이터 로드: (1050, 91)
🎯 이진분류 타겟 분포:
   정상 (0): 596개 ( 56.8%)
   질병 (1): 454개 ( 43.2%)
📈 불균형 비율: 1.31:1
   ✅ 매우 균형적! 특별한 처리 불필요

🔧 ===== 특성 선택 및 정리 (데이터 누수 방지) =====
🚨 데이터 누수 의심 특성 제거: 4개
   - medical_value_0: 상관관계 0.823 (너무 높음!)
   - medical_value_1: 상관관계 0.780 (너무 높음!)
   - medical_foot_position_0: 상관관계 nan (너무 높음!)
   - medical_foot_position_1: 상관관계 nan (너무 높음!)
   → 이런 특성들은 실제 예측 시 사용할 수 없는 정보입니다
📊 전체 특성 수 (누수 제거 후): 85
❌ 80% 이상 결측 특성 제거: 3개
   - P12_x: 98.1% 결측
   - P12_y: 98.1% 결측
   - P12_conf: 98.1% 결측

📊 이진분류 타겟과 상관관계 TOP 15 (현실적 범위):
    1. P9_y                          : 0.118 ██
    2. P3_y                          : 0.097 █
    3. P11_y                         : 0.091 █
    4. P10_y                         : 0.090 █
    5. P11_x                         : 0.084 █
    6. P10_x                         : 0.074 █
    7. P2_y                  